# Vertical Federated Unlearning (VFU): verification + slide figures

**Single notebook** for **unlearning** (sample / feature / client) and **attack-based verification** (output MIA, embedding MIA, GB-MIA, FUMA, shadow meta-model). **Figures** auto-save to **`figures_vfu_verification/`** (300 DPI PNG) for slides—see §1 (`FIG_DIR`, `save_slide_fig`) and §5b. Companion to `Part_1.ipynb`; this path uses the streamlined `SplitVFL` pipeline and **six tabular benchmarks** (§2).

---
## What “Q1” usually expects in this space

Venues such as **IEEE TIFS**, **IEEE TDSC**, **Information Fusion**, **Information Sciences**, and top ML venues care about: (i) a **clear threat model** (who sees what: logits, embeddings, gradients); (ii) **baselines** including retraining or certified methods when claims are strong; (iii) **standardized metrics** (utility drop, unlearning cost, attack success); and (iv) **honest limitations** when guarantees are empirical.

---
## Literature snapshot (2023–2025)

| Direction | Representative idea | Why it matters |
|-----------|---------------------|----------------|
| **Optimization / distillation** | VFU-KD / VFU-GA (Varshney et al., arXiv:2501.13683) | Practical unlearning for client / feature / sample targets |
| **Certified / anytime** | Theoretically certified asynchronous VFL unlearning (e.g., arXiv:2502.17081) | Bridges compliance language with formal statements |
| **Verification** | Backdoor- or certification-style checks (e.g., arXiv:2412.11476) | Unlearning is partly an **audit** problem |
| **Representation-centric** | Representation misdirection / encoder-focused forgetting (e.g., REMISVFU, arXiv:2512.10348) | Embeddings often leak more than logits |
| **Benchmarks** | Public VFU benchmarks (e.g., Vertical-Federated-Unlearning-Benchmark on GitHub) | Reduces cherry-picking and enables fair comparisons |

---
## Publishable gaps (concrete paper sketches)

1. **Multi-level verification for VFU**  
   Unify *logit-based*, *embedding-based*, and (where available) *gradient-based* membership audits under one protocol; show when KD/GA “passes” weak tests but fails stronger ones—your `Project_Report.tex` already points at this gap.

2. **Feature-unlearning with dependency structure**  
   When Party B drops a column correlated with Party A’s features, quantify **residual influence** via causal / redundancy graphs; propose unlearning + auditing that conditions on cross-party correlation.

3. **Cost–guarantee Pareto front**  
   Plot compute/communication vs. attack AUC for retrain-from-scratch, approximate methods, and (if implemented) certified variants; target **anytime** or **sequential** deletion streams.

4. **VFU under non-IID vertical splits**  
   Real institutions have different missingness and quality; study unlearning stability when passive parties have shifted feature distributions.

5. **Regulatory alignment**  
   Map methods to GDPR “erasure” narratives: *which artifacts* must be deleted (checkpoints, embedding caches, optimizer states)?

The code below builds **oracles and approximations** for all three targets (§5), summarizes them in **`df_summary`**, and runs **multi-channel sample verification** in §6 (`df_audit` + FUMA + shadow scores).

---
### PETS 2025 (Varshney et al.) — what you already use

The PETS paper evaluates VFU with **output-based membership inference** (MIA) as verification: after unlearning, forgotten samples should look like **non-members** under the attack (target AUC near **0.5**). That is a **black-box, logits-only** auditor—appropriate for the paper’s threat model but **not** the whole story in VFL, where **embeddings** (split neural representations) are an additional leakage channel.

---
### Guided workflow for *your* project (HFL-style verification → VFU)

**Step A — Fix the unlearning target:** sample / feature / client (PETS covers all three; this notebook **runs all three** in §5; MIA in §6 is defined for **sample** unlearning).

**Step B — Baseline auditor (PETS-aligned):** loss or confidence-based MIA on **final outputs** only. *Pass heuristic:* AUC on “forgotten vs held-out test” approaches 0.5 after unlearning.

**Step C — VFL-specific auditor (extends HFL-style output MIA):** same protocol but features are **concatenated bottom-model embeddings** (simulates a stronger observer who sees intermediate representations). *Document explicitly:* this is **not** always realistic for every deployment, but it stress-tests representation leakage.

**Step D — Oracle comparison:** retrain on retained data only; use as **upper bound** on what “good” forgetting looks like for metrics in B–C.

**Step E — Report a small table per dataset:** utility (acc/AUC), unlearning time, and the §6 metrics (output / embedding / GB-MIA / FUMA / shadow). **Gradient and forensic checks assume a stronger auditor** than PETS output-only MIA—state that explicitly in your report.

**Datasets:** Section 2 loads **several free tabular benchmarks** (UCI + sklearn) so you are not limited to Adult + German Credit.

## 1. Setup

In [ ]:
# Optional: pip install pandas numpy scikit-learn torch matplotlib seaborn
from pathlib import Path

import io
import time
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---- Figures for slides / report (PNG, high DPI) ---------------------------
FIG_DIR = Path("figures_vfu_verification")
FIG_DIR.mkdir(parents=True, exist_ok=True)
FIG_DPI = 300


def save_slide_fig(stem: str, tight_layout=True):
    """Save current figure to FIG_DIR / {stem}.png for Beamer/PowerPoint."""
    out = FIG_DIR / f"{stem}.png"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(out, dpi=FIG_DPI, bbox_inches="tight", facecolor="white", edgecolor="none")
    print("Saved:", out.resolve())


sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.autolayout"] = False

## 2. Free tabular datasets for VFU (multi-benchmark)

All sources are **public and credential-free** (UCI ML Repository HTTP endpoints or `sklearn`). Each dataset is one **aligned table**: rows are individuals/records, columns are features; the **label** is the prediction target. In VFL we **vertically partition columns** across parties (same row order, disjoint feature sets). Party **0** is **active** (holds labels); party **1** is **passive** in the default 2-party split below.

| Key | Rows (approx) | # raw features | Target | Source |
|-----|---------------|----------------|--------|--------|
| `adult` | ~30k | 14 | Binary: income >50K | UCI Adult |
| `german_credit` | 1000 | 20 | Binary: good vs bad credit | UCI Statlog |
| `bank_marketing` | ~45k | 16 inputs | Binary: term deposit subscribed | UCI Bank Marketing |
| `breast_cancer` | 569 | 30 | Binary: malignant vs benign | `sklearn` Wisconsin Diagnostic |
| `heart_cleveland` | ~300 | 13 | Binary: angiographic disease (4-class collapsed) | UCI Heart (Cleveland) |
| `spambase` | 4601 | 57 | Binary: spam vs not spam | UCI Spambase |

### 2.1 Feature lists (for vertical splits & unlearning)

**`adult`** — Census-style attributes; categoricals label-encoded in code.  
`age`, `workclass`, `fnlwgt`, `education`, `education_num`, `marital_status`, `occupation`, `relationship`, `race`, `sex`, `capital_gain`, `capital_loss`, `hours_per_week`, `native_country` → target **`income`** (>50K vs not).

**`german_credit`** — Credit scoring; mixed categorical + numeric (all encoded to numeric in loader).  
`checking_status`, `duration`, `credit_history`, `purpose`, `credit_amount`, `savings_status`, `employment`, `installment_rate`, `personal_status`, `other_parties`, `residence_since`, `property_magnitude`, `age`, `other_payment_plans`, `housing`, `existing_credits`, `job`, `num_dependents`, `own_telephone`, `foreign_worker` → target **`class`** (good credit vs bad).

**`bank_marketing`** — Direct marketing campaign attributes; `y` (yes/no) becomes binary label; remaining columns after drop are label-encoded if categorical. Typical columns: `age`, `job`, `marital`, `education`, `default`, `balance`, `housing`, `loan`, `contact`, `day`, `month`, `duration`, `campaign`, `pdays`, `previous`, `poutcome` → target **subscription**.

**`breast_cancer`** — 30 real-valued measurements (radius/texture/perimeter/area/smoothness/compactness/concavity/concave points/symmetry/fractal dimension, mean + error + worst). Names are in `sklearn` `feature_names`. Target: diagnosis.

**`heart_cleveland`** — Clinical: `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`, `oldpeak`, `slope`, `ca`, `thal` → target **presence of heart disease** (binary: original label >0).

**`spambase`** — 57 numeric word/char frequency attributes `f0`…`f56` (as loaded) → target **spam**.

### 2.2 Mapping to PETS-style unlearning (three targets)

| Mode | Meaning in this notebook | Default realization |
|------|--------------------------|---------------------|
| **Sample** | Remove influence of specific training rows (same ID across parties) | Retrain oracle on retained rows; VFU-GA + retain fine-tune |
| **Feature** | Remove one column contributed by the **passive** party | Drop that column everywhere; **oracle** retrain 2-party VFL on reduced table |
| **Client** | Passive party leaves; active keeps labels | **Oracle:** 1-party VFL (active features only). **Approx:** fix passive embedding to zeros; fine-tune active bottom + top (same top input dim) |

Set `VFU_DATASET` in the next cell, then rerun from that cell through Section 5 for a full run.

In [ ]:
from sklearn.datasets import load_breast_cancer


def _encode_frame(df: pd.DataFrame, skip_cols=None):
    skip = set(skip_cols or [])
    out = df.copy()
    for c in out.columns:
        if c in skip:
            continue
        if out[c].dtype == object or str(out[c].dtype) == "category":
            out[c] = LabelEncoder().fit_transform(out[c].astype(str))
    return out


def load_adult():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    columns = [
        "age", "workclass", "fnlwgt", "education", "education_num",
        "marital_status", "occupation", "relationship", "race", "sex",
        "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
    ]
    df = pd.read_csv(url, names=columns, sep=",\\s*", engine="python", na_values="?")
    df = df.dropna()
    df["income"] = (df["income"] == ">50K").astype(int)
    cat = ["workclass", "education", "marital_status", "occupation", "relationship", "race", "sex", "native_country"]
    for c in cat:
        df[c] = LabelEncoder().fit_transform(df[c])
    X = df.drop("income", axis=1)
    y = df["income"].astype(np.float32)
    return X, y


def load_german_credit():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
    columns = [
        "checking_status", "duration", "credit_history", "purpose", "credit_amount",
        "savings_status", "employment", "installment_rate", "personal_status",
        "other_parties", "residence_since", "property_magnitude", "age",
        "other_payment_plans", "housing", "existing_credits", "job",
        "num_dependents", "own_telephone", "foreign_worker", "class",
    ]
    df = pd.read_csv(url, names=columns, sep=" ")
    df["class"] = (df["class"] == 1).astype(int)
    cat = [
        "checking_status", "credit_history", "purpose", "savings_status", "employment",
        "personal_status", "other_parties", "property_magnitude", "other_payment_plans",
        "housing", "job", "own_telephone", "foreign_worker",
    ]
    for c in cat:
        df[c] = LabelEncoder().fit_transform(df[c].astype(str))
    X = df.drop("class", axis=1)
    y = df["class"].astype(np.float32)
    return X, y


def load_bank_marketing():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip"
    raw = urllib.request.urlopen(url, timeout=120).read()
    zf = zipfile.ZipFile(io.BytesIO(raw))
    with zf.open("bank-full.csv") as f:
        df = pd.read_csv(f, sep=";")
    y = (df["y"] == "yes").astype(np.float32)
    df = df.drop(columns=["y"])
    df = _encode_frame(df)
    return df, y


def load_breast_cancer():
    d = load_breast_cancer()
    X = pd.DataFrame(d.data, columns=d.feature_names)
    y = d.target.astype(np.float32)
    return X, pd.Series(y)


def load_heart_cleveland():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
    names = [
        "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
        "exang", "oldpeak", "slope", "ca", "thal", "target",
    ]
    df = pd.read_csv(url, names=names, na_values="?")
    df = df.dropna()
    y = (df["target"] > 0).astype(np.float32)
    X = df.drop(columns=["target"])
    return X, y


def load_spambase():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"
    df = pd.read_csv(url, header=None)
    y = df.iloc[:, -1].astype(np.float32)
    X = df.iloc[:, :-1]
    X.columns = [f"f{i}" for i in range(X.shape[1])]
    return X, y


def load_vfu_dataset(key: str):
    """Return (X, y, meta) with suggested training hyperparameters for small data."""
    k = key.lower().strip().replace(" ", "_")
    if k == "adult":
        X, y = load_adult()
        meta = {"key": k, "epochs_train": 45, "epochs_oracle": 45, "batch": 512, "lr": 1e-3}
    elif k == "german_credit":
        X, y = load_german_credit()
        meta = {"key": k, "epochs_train": 100, "epochs_oracle": 100, "batch": 64, "lr": 1e-3}
    elif k == "bank_marketing":
        X, y = load_bank_marketing()
        meta = {"key": k, "epochs_train": 40, "epochs_oracle": 40, "batch": 512, "lr": 1e-3}
    elif k == "breast_cancer":
        X, y = load_breast_cancer()
        meta = {"key": k, "epochs_train": 80, "epochs_oracle": 80, "batch": 64, "lr": 1e-3}
    elif k == "heart_cleveland":
        X, y = load_heart_cleveland()
        meta = {"key": k, "epochs_train": 150, "epochs_oracle": 150, "batch": 32, "lr": 5e-4}
    elif k == "spambase":
        X, y = load_spambase()
        meta = {"key": k, "epochs_train": 50, "epochs_oracle": 50, "batch": 256, "lr": 1e-3}
    else:
        raise ValueError(f"Unknown VFU_DATASET {key!r}. Choose from adult, german_credit, bank_marketing, breast_cancer, heart_cleveland, spambase.")
    meta["n_samples"] = len(y)
    meta["n_features"] = X.shape[1]
    return X, y, meta


# ---- project default: pick one benchmark ---------------------------------
VFU_DATASET = "adult"  # try: german_credit, bank_marketing, breast_cancer, heart_cleveland, spambase

X_raw, y_raw, DATASET_META = load_vfu_dataset(VFU_DATASET)
print("Dataset:", DATASET_META["key"], "|", DATASET_META["n_samples"], "samples,", DATASET_META["n_features"], "features")
print("Column names (use for FEATURE_TO_DROP):", list(X_raw.columns))
print("y distribution:", pd.Series(y_raw).value_counts().to_dict())
print("Suggested epochs/batch:", {k: DATASET_META[k] for k in ("epochs_train", "epochs_oracle", "batch", "lr")})

## 3. Vertical partition + VFL model (2 parties, active holds labels)

In [ ]:
def partition_features_equal(X: pd.DataFrame, n_parties: int, seed: int = SEED):
    rng = np.random.default_rng(seed)
    feats = list(X.columns)
    rng.shuffle(feats)
    splits = []
    k = len(feats) // n_parties
    for i in range(n_parties):
        start = i * k
        end = None if i == n_parties - 1 else (i + 1) * k
        splits.append(feats[start:end])
    return splits


class BottomNet(nn.Module):
    def __init__(self, in_dim, hid=64, emb=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.BatchNorm1d(hid),
            nn.Dropout(0.25),
            nn.Linear(hid, emb),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class TopNet(nn.Module):
    def __init__(self, in_dim, hid=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.BatchNorm1d(hid),
            nn.Dropout(0.25),
            nn.Linear(hid, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


class SplitVFL(nn.Module):
    def __init__(self, party_dims, emb=32, hid=64, dev=None):
        super().__init__()
        dev = dev or device
        self.embedding_dim = emb
        self.bottoms = nn.ModuleList([BottomNet(d, hid, emb).to(dev) for d in party_dims])
        self.top = TopNet(emb * len(party_dims), hid).to(dev)
        self.dev = dev

    def forward(self, xs, return_emb=False):
        embs = [b(x) for b, x in zip(self.bottoms, xs)]
        h = torch.cat(embs, dim=1)
        out = self.top(h)
        return (out, h) if return_emb else out

    def all_params(self):
        return list(self.parameters())


class VFLClientApprox(nn.Module):
    """Client-level approximate unlearning: passive party embedding forced to zeros; train active bottom + top."""

    def __init__(self, vfl: SplitVFL, zero_party: int = 1):
        super().__init__()
        self.vfl = vfl
        self.zero_party = zero_party
        self.embedding_dim = vfl.embedding_dim

    def forward(self, xs, return_emb=False):
        embs = []
        for i, (b, x) in enumerate(zip(self.vfl.bottoms, xs)):
            if i == self.zero_party:
                embs.append(
                    torch.zeros(x.shape[0], self.embedding_dim, device=x.device, dtype=x.dtype)
                )
            else:
                embs.append(b(x))
        h = torch.cat(embs, dim=1)
        out = self.vfl.top(h)
        return (out, h) if return_emb else out

    def freeze_party(self, party_idx: int):
        for p in self.vfl.bottoms[party_idx].parameters():
            p.requires_grad = False

    def trainable_params(self):
        return [p for p in self.vfl.parameters() if p.requires_grad]


def tensors_from_partition(X_df, y_series, feat_splits, test_size=0.25, train_idx=None, test_idx=None):
    X_parts = [X_df[cols].values.astype(np.float32) for cols in feat_splits]
    y = y_series.values.astype(np.float32)
    n = len(y)
    idx = np.arange(n)
    if train_idx is None or test_idx is None:
        tr, te = train_test_split(idx, test_size=test_size, random_state=SEED, stratify=y)
    else:
        tr, te = train_idx, test_idx
    scalers = []
    Xtr, Xte = [], []
    for Xp in X_parts:
        sc = StandardScaler()
        Xtr.append(torch.tensor(sc.fit_transform(Xp[tr]), dtype=torch.float32, device=device))
        Xte.append(torch.tensor(sc.transform(Xp[te]), dtype=torch.float32, device=device))
        scalers.append(sc)
    ytr = torch.tensor(y[tr], dtype=torch.float32, device=device).unsqueeze(1)
    yte = torch.tensor(y[te], dtype=torch.float32, device=device).unsqueeze(1)
    return Xtr, Xte, ytr, yte, tr, te, scalers


N_PARTIES = 2
splits = partition_features_equal(X_raw, N_PARTIES)
for i, s in enumerate(splits):
    print(f"Party {i} ({len(s)} feats):", s[:6], "..." if len(s) > 6 else "")

party_dims = [len(s) for s in splits]
Xtr, Xte, ytr, yte, train_idx, test_idx, scalers = tensors_from_partition(X_raw, y_raw, splits)
model = SplitVFL(party_dims).to(device)

## 4. Training helpers

In [ ]:
bce = nn.BCELoss()


def train_epochs(m, Xtr, ytr, epochs=40, lr=1e-3, bs=256, verbose=True, params=None):
    if params is None:
        params = m.trainable_params() if hasattr(m, "trainable_params") else m.all_params()
    opt = optim.Adam(params, lr=lr)
    m.train()
    n = ytr.shape[0]
    hist = []
    for ep in range(epochs):
        perm = torch.randperm(n, device=device)
        loss_acc = 0.0
        steps = 0
        for start in range(0, n, bs):
            idx = perm[start : start + bs]
            xb = [X[idx] for X in Xtr]
            yb = ytr[idx]
            opt.zero_grad()
            pred = m(xb)
            loss = bce(pred, yb)
            loss.backward()
            opt.step()
            loss_acc += loss.item()
            steps += 1
        hist.append(loss_acc / max(steps, 1))
        if verbose and (ep + 1) % 10 == 0:
            print(f"epoch {ep+1} loss {hist[-1]:.4f}")
    return hist


@torch.no_grad()
def metrics_split(m, Xte, yte):
    m.eval()
    p = m(Xte).cpu().numpy().ravel()
    yt = yte.cpu().numpy().ravel()
    yhat = (p >= 0.5).astype(int)
    acc = accuracy_score(yt, yhat)
    try:
        auc = roc_auc_score(yt, p)
    except ValueError:
        auc = float("nan")
    return acc, auc, p


t0 = time.perf_counter()
train_epochs(
    model,
    Xtr,
    ytr,
    epochs=DATASET_META["epochs_train"],
    lr=DATASET_META["lr"],
    bs=DATASET_META["batch"],
)
t_train = time.perf_counter() - t0
acc0, auc0, _ = metrics_split(model, Xte, yte)
print(f"Post-train: acc={acc0:.4f} auc={auc0:.4f} time={t_train:.1f}s")

## 5. Three-level unlearning (PETS-style targets)

Runs **sample → feature → client** on the **same train/test row indices** (see §2.2).

**5.1 Sample** — Forget random training rows; oracle retrain on retain set; VFU-GA + retain fine-tune.

**5.2 Feature** — Drop one column from the **passive** party (`PASSIVE_PARTY`, default 1). Set `FEATURE_TO_DROP` to a column name string to choose which (must be in that party’s block). Oracle: retrain 2-party VFL without that column.

**5.3 Client** — Passive party leaves. **Oracle:** 1-party VFL (active features only). **Approx:** `VFLClientApprox` (zero passive embedding; fine-tune active bottom + top).

## 5 (implementation): sample, feature, and client unlearning

The next code cell trains oracles/approximations and prints `df_summary`. Run it after Section 4.

In [ ]:
import copy


def clone_train_fresh(party_dims, X_list, y_labels, epochs, lr, batch):
    m = SplitVFL(party_dims).to(device)
    t0 = time.perf_counter()
    train_epochs(m, X_list, y_labels, epochs=epochs, lr=lr, bs=batch, verbose=False)
    return m, time.perf_counter() - t0


def gradient_ascent_forget(m, X_f, y_f, steps=80, lr=5e-4):
    opt = optim.Adam(m.all_params(), lr=lr)
    m.train()
    t0 = time.perf_counter()
    for _ in range(steps):
        opt.zero_grad()
        pred = m([x for x in X_f])
        loss = bce(pred, y_f)
        (-loss).backward()
        opt.step()
    return m, time.perf_counter() - t0


def finetune_retain(m, X_r, y_r, epochs=15, lr=1e-4, batch_size=256, params=None):
    if params is None:
        params = (
            m.trainable_params() if hasattr(m, "trainable_params") else m.all_params()
        )
    opt = optim.Adam(params, lr=lr)
    m.train()
    t0 = time.perf_counter()
    n = y_r.shape[0]
    for ep in range(epochs):
        perm = torch.randperm(n, device=device)
        for start in range(0, n, batch_size):
            idx = perm[start : start + batch_size]
            xb = [X[idx] for X in X_r]
            yb = y_r[idx]
            opt.zero_grad()
            loss = bce(m(xb), yb)
            loss.backward()
            opt.step()
    return m, time.perf_counter() - t0


_bs = DATASET_META["batch"]
_ep_o = DATASET_META["epochs_oracle"]
_lr = DATASET_META["lr"]
_et = DATASET_META["epochs_train"]
ft_after_ga = max(10, min(30, _et // 4))
ft_retain_only = max(15, min(40, _et // 3))

state_ref = copy.deepcopy(model.state_dict())
summary_rows = []

# ----- 5.1 Sample-level ----------------------------------------------------
FORGET_FRAC = 0.05
rng = np.random.default_rng(SEED)
n_tr = ytr.shape[0]
forget_mask = np.zeros(n_tr, dtype=bool)
n_forget = max(1, int(FORGET_FRAC * n_tr))
forget_pos = rng.choice(n_tr, size=n_forget, replace=False)
forget_mask[forget_pos] = True
retain_mask = ~forget_mask
print("[Sample] forget / retain:", forget_mask.sum(), retain_mask.sum())

Xtr_forget = [X[forget_mask] for X in Xtr]
y_forget = ytr[forget_mask]
Xtr_retain = [X[retain_mask] for X in Xtr]
y_retain = ytr[retain_mask]
forget_global_idx = train_idx[forget_pos]

model_sample_oracle, t_s_o = clone_train_fresh(
    party_dims, Xtr_retain, y_retain, epochs=_ep_o, lr=_lr, batch=_bs
)

model_sample_ga = SplitVFL(party_dims).to(device)
model_sample_ga.load_state_dict(copy.deepcopy(state_ref))
model_sample_ga, t_ga = gradient_ascent_forget(
    model_sample_ga, Xtr_forget, y_forget, steps=100, lr=5e-4
)
model_sample_ga, t_ft_ga = finetune_retain(
    model_sample_ga,
    Xtr_retain,
    y_retain,
    epochs=ft_after_ga,
    lr=1e-4,
    batch_size=_bs,
)
t_ga_total = t_ga + t_ft_ga

model_sample_ft = SplitVFL(party_dims).to(device)
model_sample_ft.load_state_dict(copy.deepcopy(state_ref))
model_sample_ft, t_s_ft = finetune_retain(
    model_sample_ft,
    Xtr_retain,
    y_retain,
    epochs=ft_retain_only,
    lr=1e-4,
    batch_size=_bs,
)

for label, m_, sec in [
    ("pre_unlearn (trained)", model, 0.0),
    ("oracle_retrain_retain_only", model_sample_oracle, t_s_o),
    ("vfu_ga_plus_retain_ft", model_sample_ga, t_ga_total),
    ("retain_finetune_only", model_sample_ft, t_s_ft),
]:
    acc, auc, _ = metrics_split(m_, Xte, yte)
    summary_rows.append(
        {"unlearn_type": "sample", "variant": label, "acc": acc, "auc": auc, "seconds": sec}
    )

# ----- 5.2 Feature-level ---------------------------------------------------
PASSIVE_PARTY = 1
FEATURE_TO_DROP = None  # set to e.g. "age" if that column is on the passive party
if FEATURE_TO_DROP is None:
    feature_to_drop = splits[PASSIVE_PARTY][0]
else:
    feature_to_drop = FEATURE_TO_DROP
    assert feature_to_drop in splits[PASSIVE_PARTY], "FEATURE_TO_DROP must be in passive party block"

splits_feat = [[c for c in party_cols if c != feature_to_drop] for party_cols in splits]
assert all(len(cols) > 0 for cols in splits_feat), "A party would have zero features after drop"

X_feat_df = X_raw.drop(columns=[feature_to_drop])
party_dims_sf = [len(c) for c in splits_feat]
Xtr_sf, Xte_sf, ytr_sf, yte_sf, _, _, _ = tensors_from_partition(
    X_feat_df, y_raw, splits_feat, train_idx=train_idx, test_idx=test_idx
)

model_feature_oracle, t_f_o = clone_train_fresh(
    party_dims_sf, Xtr_sf, ytr_sf, epochs=_ep_o, lr=_lr, batch=_bs
)

acc_full_baseline, auc_full_baseline, _ = metrics_split(model, Xte, yte)
acc_feat_o, auc_feat_o, _ = metrics_split(model_feature_oracle, Xte_sf, yte_sf)

summary_rows.append(
    {
        "unlearn_type": "feature",
        "variant": f"baseline_full_model (before drop {feature_to_drop})",
        "acc": acc_full_baseline,
        "auc": auc_full_baseline,
        "seconds": 0.0,
    }
)
summary_rows.append(
    {
        "unlearn_type": "feature",
        "variant": f"oracle_retrain_without_{feature_to_drop}",
        "acc": acc_feat_o,
        "auc": auc_feat_o,
        "seconds": t_f_o,
    }
)

print(f"[Feature] dropped column (party {PASSIVE_PARTY}): {feature_to_drop!r}")

# ----- 5.3 Client-level ----------------------------------------------------
splits_active_only = [splits[0]]
X_active_df = X_raw[splits_active_only[0]]
party_dims_1p = [len(splits_active_only[0])]
Xtr_1p, Xte_1p, ytr_1p, yte_1p, _, _, _ = tensors_from_partition(
    X_active_df, y_raw, splits_active_only, train_idx=train_idx, test_idx=test_idx
)

model_client_oracle, t_c_o = clone_train_fresh(
    party_dims_1p, Xtr_1p, ytr_1p, epochs=_ep_o, lr=_lr, batch=_bs
)

base_client = SplitVFL(party_dims).to(device)
base_client.load_state_dict(copy.deepcopy(state_ref))
model_client_approx = VFLClientApprox(base_client, zero_party=1)
model_client_approx.freeze_party(1)
t0 = time.perf_counter()
train_epochs(
    model_client_approx,
    Xtr,
    ytr,
    epochs=ft_retain_only,
    lr=1e-4,
    bs=_bs,
    verbose=False,
    params=model_client_approx.trainable_params(),
)
t_c_apx = time.perf_counter() - t0

acc_cl_o, auc_cl_o, _ = metrics_split(model_client_oracle, Xte_1p, yte_1p)
acc_cl_apx, auc_cl_apx, _ = metrics_split(model_client_approx, Xte, yte)

summary_rows.append(
    {
        "unlearn_type": "client",
        "variant": "baseline_full_2party_model",
        "acc": acc_full_baseline,
        "auc": auc_full_baseline,
        "seconds": 0.0,
    }
)
summary_rows.append(
    {
        "unlearn_type": "client",
        "variant": "oracle_retrain_active_party_only",
        "acc": acc_cl_o,
        "auc": auc_cl_o,
        "seconds": t_c_o,
    }
)
summary_rows.append(
    {
        "unlearn_type": "client",
        "variant": "approx_zero_passive_emb_finetune",
        "acc": acc_cl_apx,
        "auc": auc_cl_apx,
        "seconds": t_c_apx,
    }
)

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

## 5b. Figures: utility vs. unlearning cost (for slides)

All PNGs go to **`figures_vfu_verification/`** (300 DPI). Filenames start with the dataset key (e.g. `adult_01_...`). Re-run after changing `VFU_DATASET`.


In [ ]:
tag = DATASET_META["key"]

samp = df_summary[df_summary["unlearn_type"] == "sample"].reset_index(drop=True)
if len(samp):
    fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
    sns.barplot(data=samp, x="variant", y="acc", ax=ax[0], color="#4682b4")
    ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=24, ha="right")
    ax[0].set_title("Sample unlearning — accuracy (test)")
    ax[0].set_ylim(0, 1)
    sns.barplot(data=samp, x="variant", y="auc", ax=ax[1], color="#2e8b57")
    ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=24, ha="right")
    ax[1].set_title("Sample unlearning — ROC-AUC (test)")
    ax[1].set_ylim(0, 1)
    save_slide_fig(f"{tag}_01_sample_unlearn_acc_auc")
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(8.5, 3.8))
    sns.barplot(data=samp, x="variant", y="seconds", ax=ax2, color="#e6550d")
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=24, ha="right")
    ax2.set_title("Sample unlearning — compute time (s)")
    save_slide_fig(f"{tag}_02_sample_unlearn_wallclock")
    plt.show()

fc = df_summary[df_summary["unlearn_type"].isin(["feature", "client"])].reset_index(drop=True)
if len(fc):
    h = max(3.2, 0.42 * len(fc) + 1.5)
    fig3, ax3 = plt.subplots(figsize=(9, h))
    sns.barplot(data=fc, y="variant", x="acc", hue="unlearn_type", ax=ax3)
    ax3.set_xlim(0, 1)
    ax3.set_title("Feature and client unlearning — test accuracy")
    ax3.legend(title="Type", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_slide_fig(f"{tag}_06_feature_client_accuracy")
    plt.show()


## 6. Verification: what each method does (and how it relates to unlearning)

### 6.1 Membership inference (MIA) — core idea
Given a **trained model** and a **candidate sample** $(x, y)$, a MIA tries to guess whether that sample was **in the training set** or not. It is **not** the same as test accuracy: the attacker uses **side signals** (loss, confidence, gradients, internal states) that often differ between members and non-members. **ROC-AUC** summarizes how well the attacker separates the two classes; **0.5** is random guessing.

**Important:** In *unlearning verification*, we reuse MIA-style machinery with a **twist**: we want **forgotten** samples to **resemble non-members** (or retain-only behavior) under the attack—so **lower attack success** on the forget set often indicates **better** erasure *under that auditor*.

### 6.2 Methods implemented here (aligned with `Part_1.ipynb`)

| Method | What the “attacker” sees | What we compute | Unlearning readout (heuristic) |
|--------|-------------------------|-----------------|--------------------------------|
| **Output / logit MIA** | Final probability $\hat{p}(y\mid x)$ only | Per-sample BCE loss $\ell = -\bigl(y\log \hat{p} + (1-y)\log(1-\hat{p})\bigr)$; AUC using score $-\ell$ (members tend to have **smaller** loss) | After sample unlearning, forget-set AUC should move **toward 0.5** vs held-out test (PETS-style). |
| **Embedding MIA** | Concatenated **bottom-model embeddings** (VFL-specific) | Logistic regression on embeddings to separate forget batch vs test batch | Stress-tests **representation** leakage; can stay high even when logits look “safe”. |
| **GB-MIA (gradient-based)** | **Per-sample gradients** w.r.t. all parameters (white-box) | For each sample: one forward + backward; vector of **gradient norms / stats** (~20 dims, as in Part 1); **Random Forest** trains member vs non-member; report **attack AUC** on held-out split | Stronger auditor: if AUC stays high after unlearning, **parameter-level** traces may remain. **Expensive** (we cap ~60 samples per side). |
| **FUMA (forensic)** | Full model **before** and **after** unlearning (checkpoint comparison) | For each row: load **pre**-unlearn weights → pred, loss, grad; load **post**-unlearn weights → same; features include $\lvert p_{\text{after}}-p_{\text{before}}\rvert$, $\Delta$loss, grad-norm ratio, $1-\cos(\mathbf{g}_b,\mathbf{g}_a)$, etc.; **GradientBoosting** learns “unlearned vs retained”; **detection rate** = fraction of **forget** rows classified as unlearned | **Lower** detection on forget rows ⇒ forensic traces harder to spot (Part 1 narrative). |
| **Shadow models** | Only **public behavior** on a fixed **probe** subset + **weight statistics** of shadow models | Train several **shadow** `SplitVFL` models on **retain-only** vs **full** train; meta–RF on [weight stats ‖ probe-output stats]; report **P(“full-data-like”)** for your target models + meta-AUC on shadows | After good unlearning, the unlearned model should look **more like retain-only** shadows → **lower** P(full). |

### 6.3 Recent trends in VFU (2024–2026) — for your report
- **Certified / anytime unlearning** in VFL (formal removal guarantees, asynchronous requests).
- **Representation-level forgetting** (encoder collapse / orthogonal projection / REMISVFU-style) when logits pass audits but embeddings fail.
- **Standardized benchmarks** (e.g. public VFU benchmark suites) and **multi-channel verification** exactly like this notebook: output → embedding → gradient → forensic → shadow.
- **Regulatory / systems view:** what must be deleted (checkpoints, cached embeddings, optimizer state), not only weights.
- **Feature entanglement:** unlearning one party’s column when features are correlated across parties.

### 6.4 Code cell below
Runs **sample-level** verification on `model`, `model_sample_oracle`, `model_sample_ga`, `model_sample_ft`. Requires prior cells (§5) defining `forget_mask`, `forget_pos`, `clone_train_fresh`, etc.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# ---------------------------------------------------------------------------
# What each method does (see markdown §6 for full threat model)
# ---------------------------------------------------------------------------
# Output / logit MIA:  score = -BCE(y, p); members usually have lower loss.
# Embedding MIA:     attacker sees concat bottom embeddings; trains logistic.
# GB-MIA:              per-sample backward; vector of gradient norms/stats; RF member vs non-member.
# FUMA:                compare model frozen at pre-unlearn vs post-unlearn on same inputs;
#                      build vector (|Δpred|, Δloss, grad norm ratio, cosine grad change, ...);
#                      GB classifier: unlearned row vs retained row (detection rate on forget set).
# Shadow:              train small shadow SplitVFLs on retain-only vs full train; meta-RF on
#                      weight statistics + probe-set prediction stats; score target models' P(full).


def collect_signals(m, X_list, y_labels):
    was_training = m.training if hasattr(m, "training") else False
    m.eval()
    with torch.no_grad():
        pred, emb = m(X_list, return_emb=True)
        p = pred.clamp(1e-6, 1 - 1e-6)
        y = y_labels
        bce_elem = -(y * p.log() + (1 - y) * (1 - p).log())
    if was_training:
        m.train()
    return bce_elem.cpu().numpy().ravel(), emb.cpu().numpy()


def mia_auc_logit_loss(model_under_test, X_mem, y_mem, X_non, y_non):
    l_mem, _ = collect_signals(model_under_test, X_mem, y_mem)
    l_non, _ = collect_signals(model_under_test, X_non, y_non)
    y_attack = np.array([1] * len(l_mem) + [0] * len(l_non))
    try:
        return roc_auc_score(y_attack, -np.concatenate([l_mem, l_non]))
    except ValueError:
        return float("nan")


def mia_auc_embedding(model_under_test, X_mem, y_mem, X_non, y_non, seed=0):
    _, E_mem = collect_signals(model_under_test, X_mem, y_mem)
    _, E_non = collect_signals(model_under_test, X_non, y_non)
    X = np.vstack([E_mem, E_non])
    y = np.array([1] * len(E_mem) + [0] * len(E_non))
    tr_i, te_i = train_test_split(
        np.arange(len(y)), test_size=0.35, random_state=seed, stratify=y
    )
    clf = LogisticRegression(max_iter=200, class_weight="balanced")
    clf.fit(X[tr_i], y[tr_i])
    s = clf.predict_proba(X[te_i])[:, 1]
    return roc_auc_score(y[te_i], s)


def _extract_grad_feature_vector(model: SplitVFL):
    feats = []
    all_grads = []
    for bottom in model.bottoms:
        for param in bottom.parameters():
            if param.grad is not None:
                g = param.grad.detach().cpu().numpy().ravel()
                all_grads.append(g)
    for param in model.top.parameters():
        if param.grad is not None:
            g = param.grad.detach().cpu().numpy().ravel()
            all_grads.append(g)
    if not all_grads:
        return np.zeros(20)
    full = np.concatenate(all_grads)
    l2 = float(np.linalg.norm(full))
    feats.append(l2)
    feats.append(float(np.linalg.norm(full, ord=1)))
    feats.extend(
        [
            float(np.mean(full)),
            float(np.std(full)),
            float(np.max(np.abs(full))),
            float(np.min(np.abs(full[np.abs(full) > 1e-12]))) if np.any(np.abs(full) > 1e-12) else 0.0,
        ]
    )
    for q in (25, 50, 75, 95):
        feats.append(float(np.percentile(np.abs(full), q)))
    feats.append(float(np.mean(np.abs(full) < 1e-6)))
    for bottom in model.bottoms[:2]:
        layer = []
        for param in bottom.parameters():
            if param.grad is not None:
                layer.append(param.grad.detach().cpu().numpy().ravel())
        feats.append(float(np.linalg.norm(np.concatenate(layer))) if layer else 0.0)
    topg = []
    for param in model.top.parameters():
        if param.grad is not None:
            topg.append(param.grad.detach().cpu().numpy().ravel())
    feats.append(float(np.linalg.norm(np.concatenate(topg))) if topg else 0.0)
    side = int(np.sqrt(len(full)))
    if side > 2:
        try:
            gm = full[: side * side].reshape(side, side)
            s = np.linalg.svd(gm, compute_uv=False)
            feats.extend([float(s[i]) if i < len(s) else 0.0 for i in range(3)])
        except Exception:
            feats.extend([0.0, 0.0, 0.0])
    else:
        feats.extend([0.0, 0.0, 0.0])
    if l2 > 0:
        u = full / l2
        feats.extend([float(np.mean(u)), float(np.std(u))])
    else:
        feats.extend([0.0, 0.0])
    return np.array(feats[:20], dtype=np.float64)


def compute_sample_gradients_split(model: SplitVFL, X_parties, y, max_samples=80):
    n = min(int(y.shape[0]), max_samples)
    crit = nn.BCELoss()
    rows = []
    for bottom in model.bottoms:
        bottom.eval()
    model.top.eval()
    for i in range(n):
        for bottom in model.bottoms:
            bottom.zero_grad()
        model.top.zero_grad()
        Xs = [X[i : i + 1] for X in X_parties]
        ys = y[i : i + 1]
        out = model(Xs)
        loss = crit(out, ys)
        loss.backward()
        rows.append(_extract_grad_feature_vector(model))
    return np.array(rows)


def train_gb_mia_attack(member_g, non_g):
    Xa = np.vstack([member_g, non_g])
    ya = np.array([1] * len(member_g) + [0] * len(non_g))
    Xa = np.nan_to_num(Xa, nan=0, posinf=1e6, neginf=-1e6)
    Xtr, Xte, ytr, yte = train_test_split(
        Xa, ya, test_size=0.25, random_state=SEED, stratify=ya
    )
    rf = RandomForestClassifier(
        n_estimators=80, max_depth=8, random_state=SEED, n_jobs=-1
    )
    rf.fit(Xtr, ytr)
    pr = rf.predict_proba(Xte)[:, 1]
    return float(roc_auc_score(yte, pr)), rf


def forensic_features_one_sample(model: SplitVFL, Xs, ys, state_before, state_current):
    crit = nn.BCELoss()
    current = copy.deepcopy(state_current)
    model.load_state_dict(state_before)
    for b in model.bottoms:
        b.eval()
    model.top.eval()
    out_b = model(Xs)
    with torch.no_grad():
        pb = out_b.detach().clone()
    for b in model.bottoms:
        b.zero_grad()
    model.top.zero_grad()
    lb = crit(model(Xs), ys)
    lb.backward()
    gb = torch.cat(
        [p.grad.detach().flatten() for b in model.bottoms for p in b.parameters() if p.grad is not None]
        + [p.grad.detach().flatten() for p in model.top.parameters() if p.grad is not None]
    )
    model.load_state_dict(current)
    for b in model.bottoms:
        b.eval()
    model.top.eval()
    out_a = model(Xs)
    pa = out_a.detach()
    for b in model.bottoms:
        b.zero_grad()
    model.top.zero_grad()
    la = crit(model(Xs), ys)
    la.backward()
    ga = torch.cat(
        [p.grad.detach().flatten() for b in model.bottoms for p in b.parameters() if p.grad is not None]
        + [p.grad.detach().flatten() for p in model.top.parameters() if p.grad is not None]
    )
    nb = torch.norm(gb).item() + 1e-8
    na = torch.norm(ga).item() + 1e-8
    cos = float(torch.dot(gb, ga) / (nb * na)) if gb.numel() == ga.numel() else 0.0
    return np.array(
        [
            abs(float(pb.item()) - float(pa.item())),
            float(pa.item() - pb.item()),
            float(la.item() - lb.item()),
            na / nb,
            1.0 - cos,
            float(torch.mean((torch.sign(gb) != torch.sign(ga)).float()))
            if gb.numel() == ga.numel()
            else 0.0,
        ],
        dtype=np.float64,
    )


def fuma_quick_train_and_rate(
    model_after: SplitVFL, Xtr, ytr, forget_positions, state_before, max_n=120
):
    state_after = copy.deepcopy(model_after.state_dict())
    retain_positions = np.setdiff1d(np.arange(ytr.shape[0]), forget_positions)
    nu = min(max_n, len(forget_positions))
    nr = min(max_n, len(retain_positions))
    if nu < 5 or nr < 5:
        return float("nan"), float("nan"), None
    ui = np.random.choice(forget_positions, nu, replace=False)
    ri = np.random.choice(retain_positions, nr, replace=False)

    def collect(idxs):
        F = []
        for j in idxs:
            Xs = [X[j : j + 1] for X in Xtr]
            ys = ytr[j : j + 1]
            F.append(forensic_features_one_sample(model_after, Xs, ys, state_before, state_after))
        return np.stack(F)

    Xu, Yu = collect(ui), collect(ri)
    Xa = np.vstack([Xu, Yu])
    ya = np.array([1] * len(Xu) + [0] * len(Yu))
    Xa = np.nan_to_num(Xa, nan=0, posinf=1e6, neginf=-1e6)
    clf = GradientBoostingClassifier(
        n_estimators=60, max_depth=3, random_state=SEED
    )
    Xtr_f, Xte_f, ytr_f, yte_f = train_test_split(
        Xa, ya, test_size=0.2, random_state=SEED, stratify=ya
    )
    clf.fit(Xtr_f, ytr_f)
    try:
        auc_tr = float(
            roc_auc_score(yte_f, clf.predict_proba(Xte_f)[:, 1])
        )
    except ValueError:
        auc_tr = float("nan")
    # detection rate on held-out forget
    te = min(80, len(forget_positions))
    ti = np.random.choice(forget_positions, te, replace=False)
    Xt = collect(ti)
    Xt = np.nan_to_num(Xt, nan=0, posinf=1e6, neginf=-1e6)
    pred = clf.predict(Xt)
    det = float(np.mean(pred == 1))
    model_after.load_state_dict(state_after)
    return det, auc_tr, clf


def flatten_param_stats(model: SplitVFL):
    v = []
    for p in model.parameters():
        d = p.detach().cpu().numpy().ravel()
        if d.size == 0:
            continue
        v.extend(
            [
                float(d.mean()),
                float(d.std()),
                float(np.linalg.norm(d)),
                float(np.percentile(d, 10)),
                float(np.percentile(d, 90)),
            ]
        )
    return np.array(v, dtype=np.float64)


def behavior_stats(model: SplitVFL, Xp, yp):
    model.eval()
    with torch.no_grad():
        pr = model(Xp).cpu().numpy().ravel()
    return np.array(
        [
            float(np.mean(pr)),
            float(np.std(pr)),
            float(np.mean(pr > 0.5)),
            float(np.mean(np.abs(pr - 0.5))),
        ],
        dtype=np.float64,
    )


def run_shadow_meta(
    party_dims, Xtr, ytr, forget_pos, epochs_s, lr, batch, n_shadow=3, probe_n=400
):
    all_idx = np.arange(ytr.shape[0])
    retain_idx = np.setdiff1d(all_idx, forget_pos)
    feats_r, feats_f = [], []
    ep = max(12, epochs_s // 3)
    rng = np.random.default_rng(SEED)
    for _ in range(n_shadow):
        m_r, _ = clone_train_fresh(
            party_dims,
            [X[retain_idx] for X in Xtr],
            ytr[retain_idx],
            epochs=ep,
            lr=lr,
            batch=batch,
        )
        full_sub = rng.choice(
            all_idx, max(50, int(0.9 * len(all_idx))), replace=False
        )
        m_full, _ = clone_train_fresh(
            party_dims,
            [X[full_sub] for X in Xtr],
            ytr[full_sub],
            epochs=ep,
            lr=lr,
            batch=batch,
        )
        pi = rng.choice(all_idx, min(probe_n, len(all_idx)), replace=False)
        Xp = [X[pi] for X in Xtr]
        yp = ytr[pi]
        feats_r.append(
            np.concatenate([flatten_param_stats(m_r), behavior_stats(m_r, Xp, yp)])
        )
        feats_f.append(
            np.concatenate([flatten_param_stats(m_full), behavior_stats(m_full, Xp, yp)])
        )
    Xmeta = np.vstack([feats_r, feats_f])
    ymeta = np.array([0] * len(feats_r) + [1] * len(feats_f))
    Xmeta = np.nan_to_num(Xmeta, nan=0, posinf=1e6, neginf=-1e6)
    meta = RandomForestClassifier(
        n_estimators=80, max_depth=8, random_state=SEED, class_weight="balanced"
    )
    meta.fit(Xmeta, ymeta)
    pr_shadow = meta.predict_proba(Xmeta)[:, 1]
    shadow_auc = float(roc_auc_score(ymeta, pr_shadow))

    def score_model(m):
        z = np.concatenate([flatten_param_stats(m), behavior_stats(m, Xp, yp)]).reshape(1, -1)
        z = np.nan_to_num(z, nan=0, posinf=1e6, neginf=-1e6)
        return float(meta.predict_proba(z)[0, 1])

    return shadow_auc, score_model, Xp, yp


# ----- Sample sets (same as before) -----------------------------------------
n_m = y_forget.shape[0]
sub_te = torch.randperm(yte.shape[0], device=device)[: n_m * 2]
X_non = [X[sub_te] for X in Xte]
y_non = yte[sub_te]
X_mem = Xtr_forget
y_mem = y_forget

remain_idx = np.where(~forget_mask)[0]
n_sub = min(80, len(remain_idx), n_m)
rng = np.random.default_rng(SEED)
mem_sub = rng.choice(n_m, min(n_sub, n_m), replace=False)
non_sub = rng.choice(len(X_non[0]), min(n_sub, len(X_non[0])), replace=False)
X_mem_s = [X[mem_sub] for X in X_mem]
y_mem_s = y_mem[mem_sub]
X_non_s = [X[non_sub] for X in X_non]
y_non_s = y_non[non_sub]

audit_rows = []
for name, m_ in [
    ("pre_unlearn", model),
    ("sample_oracle", model_sample_oracle),
    ("sample_vfu_ga+retain_ft", model_sample_ga),
    ("sample_retain_finetune_only", model_sample_ft),
]:
    row = {
        "method": name,
        "mia_logit_auc": mia_auc_logit_loss(m_, X_mem, y_mem, X_non, y_non),
        "mia_embedding_auc": mia_auc_embedding(m_, X_mem, y_mem, X_non, y_non),
    }
    mg = compute_sample_gradients_split(m_, X_mem_s, y_mem_s, max_samples=min(60, len(mem_sub)))
    ng = compute_sample_gradients_split(m_, X_non_s, y_non_s, max_samples=min(60, len(non_sub)))
    if len(mg) >= 10 and len(ng) >= 10:
        g_auc, _ = train_gb_mia_attack(mg, ng)
        row["gb_mia_attack_auc"] = g_auc
    else:
        row["gb_mia_attack_auc"] = float("nan")
    audit_rows.append(row)

df_audit = pd.DataFrame(audit_rows)
print("=== Sample-level verification (Part_1-style) ===")
print(df_audit.to_string(index=False))

state_before_unlearn = copy.deepcopy(model.state_dict())
det_fuma, auc_fuma, _ = fuma_quick_train_and_rate(
    model_sample_ga, Xtr, ytr, forget_pos, state_before_unlearn, max_n=100
)
print("\n=== FUMA (forensic: pre vs post unlearn on same inputs) ===")
print(
    f"  Trained on unlearned vs retained rows; detection rate on held-out FORGET rows: {det_fuma:.4f}"
)
print(f"  (Lower ≈ forget rows look like retained; GB train AUC on train split: {auc_fuma:.4f})")

ds_tag = DATASET_META["key"]
fig_fuma, ax_fu = plt.subplots(figsize=(5.5, 3.2))
ax_fu.barh(
    ["Forget rows → 'unlearned' class"],
    [det_fuma],
    color="#6a51a3",
    height=0.45,
)
ax_fu.set_xlabel("FUMA detection rate (lower is better for hiding forget set)")
ax_fu.set_xlim(0, 1)
ax_fu.set_title(f"FUMA — {ds_tag}")
save_slide_fig(f"{ds_tag}_05_fuma_forget_detection_rate")
plt.show()

sh_auc, score_fn, X_probe, y_probe = run_shadow_meta(
    party_dims,
    Xtr,
    ytr,
    forget_pos,
    DATASET_META["epochs_oracle"],
    DATASET_META["lr"],
    DATASET_META["batch"],
    n_shadow=3,
)
p_pre = score_fn(model)
p_post = score_fn(model_sample_oracle)
p_ga = score_fn(model_sample_ga)
print("\n=== Shadow meta-classifier P('full-data-like') ===")
print(f"  Meta AUC on shadow models only (sanity): {sh_auc:.4f}")
print(f"  Pre-unlearn model:        {p_pre:.4f}")
print(f"  Sample oracle (retrain): {p_post:.4f}")
print(f"  After VFU-GA pipeline:    {p_ga:.4f}")
print("  (Ideal unlearn: post-oracle near retain shadows → lower P(full).)")

fig_sh, ax_sh = plt.subplots(figsize=(6.8, 4.2))
ax_sh.bar(
    ["Pre-unlearn", "Oracle retrain", "VFU-GA + FT"],
    [p_pre, p_post, p_ga],
    color=["#636363", "#31a354", "#fd8d3c"],
)
ax_sh.axhline(0.5, color="k", linestyle="--", linewidth=1, alpha=0.45)
ax_sh.set_ylabel('Shadow meta-model P("full-data-like")')
ax_sh.set_title(f"Shadow models — {ds_tag}")
plt.xticks(rotation=18, ha="right")
save_slide_fig(f"{ds_tag}_04_shadow_meta_prob_full")
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(12, 3.8))
sns.barplot(data=df_audit, x="method", y="mia_logit_auc", ax=ax[0])
ax[0].set_title("Output (loss) MIA AUC")
ax[0].tick_params(axis="x", rotation=20)
sns.barplot(data=df_audit, x="method", y="mia_embedding_auc", ax=ax[1])
ax[1].set_title("Embedding MIA AUC")
ax[1].tick_params(axis="x", rotation=20)
sns.barplot(data=df_audit, x="method", y="gb_mia_attack_auc", ax=ax[2])
ax[2].set_title("GB-MIA AUC (grad features)")
ax[2].tick_params(axis="x", rotation=20)
save_slide_fig(f"{ds_tag}_03_mia_output_embedding_gb_mia")
plt.show()


## 7. How to turn this into a Q1 submission (checklist)

1. **Problem statement**: Specify unlearning target (sample / feature / client)—**`df_summary`** is a template table—plus adversary knowledge (black-box logits vs. white-box embeddings vs. gradients).
2. **Baselines**: Oracle retrain, your VFU-GA/KD implementations from `Part_1.ipynb`, and recent arXiv methods you reproduce via the [Vertical-Federated-Unlearning-Benchmark](https://github.com/bryanhx/Vertical-Federated-Unlearning-Benchmark) where possible.
3. **Metrics table**: Utility (acc/AUC/F1), **forget-set** behavior (e.g., confidence on forgotten labels), **attack AUC** under multiple auditors, **time/communication**.
4. **Ablations**: Forget ratio, party count, embedding dimension, sequential requests, label imbalance.
5. **Theory or certificates**: If targeting TIFS/TDSC, pair empirical results with formal statements (even partial) or tight limitations.

---
### Suggested high-impact title patterns
- *“Hierarchical Verification for Vertical Federated Unlearning: When Logits Lie”*
- *“Anytime Feature Withdrawal in VFL: Cost–Privacy Tradeoffs under GDPR”*
- *“Certified vs. Approximate VFU: A Unified Empirical Framework”*

---
**References (open access / preprints)**  
- Varshney et al., *Unlearning Clients, Features and Samples in Vertical Federated Learning*, [arXiv:2501.13683](https://arxiv.org/abs/2501.13683)  
- *Forgetting Any Data at Any Time* (certified VFL unlearning), [arXiv:2502.17081](https://arxiv.org/abs/2502.17081)  
- *Vertical Federated Unlearning via Backdoor Certification*, [arXiv:2412.11476](https://arxiv.org/abs/2412.11476)  
- REMISVFU, [arXiv:2512.10348](https://arxiv.org/abs/2512.10348)